# Lab 19: Keyword Spotting and Wake-Word Detection

**Duration:** 3 hours 30 minutes. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A voice assistant must not send every sound to a big speech model. It waits for a **wake word** (like *Hey Jarvis*), and only then listens to a command. Finding a few chosen words in a stream of sound is called **keyword spotting**. In this lab you use a **pretrained** wake-word model (openWakeWord) that is small enough to listen all the time, see its score for every 80 milliseconds of sound, and tune the threshold. Then you build the milestone of the speech labs: a **voice-controlled assistant** that wakes up on the wake word, listens to a command, understands it with speech to text (Lab 17), does a fixed action, and answers with a voice (Lab 18). There is a **push-to-talk** button if the wake word does not work in your room.

**By the end of this lab you can:**
1. Explain why a small keyword spotter listens all the time and a large speech-to-text model does not.
2. Run a pretrained wake-word model and read its score for every 80 ms of sound.
3. Tune the threshold and explain false alarms and missed words.
4. Build a voice assistant with the steps wake word, listen, understand, act and speak.
5. Test the assistant and name the cases where it fails.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Privacy.** The microphone listens all the time while the listener runs. The sound stays on the Jetson. The assistant keeps a text log of what it heard. Do not use it on people who do not know about it, and delete the log when you are done.

## 2. Background

**Why a wake word?** Speech to text (Lab 17) is large. It needs a few seconds and a few hundred MB for every sentence. Running it all day on every sound would waste power and listen to everything. A **keyword spotter** is tiny. It answers one question only: *did I hear my wake word just now?* It runs in a few milliseconds on the CPU and can run all the time. The big model wakes up only after the wake word.

| | Speech to text (Lab 17) | Keyword spotter (this lab) |
|---|---|---|
| Question | What was said? | Was it my wake word? |
| Words | Any word | A few chosen words |
| Size and speed | Large, seconds | Tiny, milliseconds |
| Used for | Understanding a sentence | Waking up, simple triggers |

**Pretrained.** In this lab you do not train a model. **openWakeWord** gives pretrained wake-word models: `hey_jarvis`, `alexa`, `hey_mycroft`, `hey_rhasspy`, `timer` and `weather`. They were trained on a lot of synthetic and real speech. In Lab 33 you train a wake word of your own.

**How it listens.** The sound arrives at 16 000 numbers per second. The model reads it in pieces of **1280 numbers (80 ms)**. For every piece it gives each wake word a **score** between 0 and 1. A score near 1 means *I am sure I heard it*. Your program compares the score with a **threshold**.

| Setting | If you raise it | If you lower it |
|---|---|---|
| `THRESHOLD` | Fewer false alarms, but more missed wake words | More wake words found, but more false alarms |
| `COOLDOWN` | Fewer repeats | Faster repeated wake-ups |

**The assistant, in five steps.** A voice assistant repeats the same steps every time. Compare them with the vision steps of Lab 16.

| Step | What happens | Tool |
|---|---|---|
| 1. Wake | Wait for the wake word | openWakeWord (this lab) |
| 2. Listen | Record until you are quiet | `listen_until_silence` |
| 3. Understand | Speech to text | Whisper (Lab 17) |
| 4. Act | Do a fixed action (light on, tell the time ...) | `reply_to` |
| 5. Speak | Say the answer | Piper (Lab 18) |

This assistant has **fixed commands**. It does not use a language model. Lab 20 adds a language model, and Lab 22 joins everything.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(19)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (25 min)

### Step 1: Settings
`WAKE_WORDS` are the wake words to use. `LIMIT_DB` is the loudness that counts as sound (see Lab 17). Your microphone was set in Lab 17 (it is remembered). Run the cell.

In [ ]:
# ---------------- Settings you can change ----------------
WAKE_WORDS = ["hey_jarvis"]    # choose from: hey_jarvis, alexa, hey_mycroft, hey_rhasspy, timer, weather
THRESHOLD = 0.50               # the score (0 to 1) that counts as "I heard it"
COOLDOWN = 2.0                 # seconds before the wake word is accepted again
LIMIT_DB = -38                 # louder than this counts as sound (the same as in Lab 17)
STT_SIZE = "base.en"           # the Whisper model of Lab 17
# ----------------------------------------------------------
import csv
import glob
import hashlib
import os
import subprocess
import sys
import threading
import time
from datetime import datetime

import numpy as np

import labview

RATE = 16000
FRAME = 1280                   # 80 ms of sound: the piece that the wake-word model reads
print("Wake words:", WAKE_WORDS, "| microphone:", labview.mic_get())

### Step 2: Install openWakeWord and Load a Model
The first cell installs what is missing (it needs the internet, about 1 minute). The second cell downloads the model files of your wake words once and loads them.

In [ ]:
import importlib


def need(package, module):
    try:
        importlib.import_module(module)
        print(f"{package:<14} is installed")
    except ImportError:
        print(f"{package:<14} installing ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)


for package, module in (("scikit-learn", "sklearn"), ("scipy", "scipy"), ("tqdm", "tqdm"), ("requests", "requests"), ("onnxruntime", "onnxruntime")):
    need(package, module)
try:
    import openwakeword
    print("openwakeword  is installed")
except ImportError:
    print("openwakeword  installing ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "openwakeword"], check=True)
    import openwakeword

In [ ]:
import openwakeword
from openwakeword.model import Model

for attempt in range(3):                               # a slow network can fail once; try again
    try:
        openwakeword.utils.download_models(WAKE_WORDS)
        break
    except Exception as error:
        print("Download problem, trying again:", str(error)[:80])
        time.sleep(2)

oww = Model(wakeword_models=WAKE_WORDS, inference_framework="onnx")
print("Loaded wake words:", list(oww.models))

### Step 3: A Detector That Reads the Sound in Pieces
The model reads **exactly 1280 numbers** at a time. The sound from the microphone arrives in pieces of any size. The class `WakeDetector` keeps the rest of the sound until the next piece, and gives back the **best score** of each wake word in the piece. Read it, then run it.

In [ ]:
class WakeDetector:
    def __init__(self, model):
        self.model = model
        self.rest = np.zeros(0, np.int16)

    def feed(self, sound):
        """sound: numbers between -1 and 1. Returns {wake word: best score in this piece of sound}."""
        pcm = np.concatenate([self.rest, (np.clip(sound, -1, 1) * 32767).astype(np.int16)])
        best = {}
        whole = len(pcm) // FRAME * FRAME
        for start in range(0, whole, FRAME):
            for name, score in self.model.predict(pcm[start:start + FRAME]).items():
                best[name] = max(best.get(name, 0.0), float(score))
        self.rest = pcm[whole:]
        return best

    def reset(self):
        self.model.reset()
        self.rest = np.zeros(0, np.int16)


def level_db(x):
    """The loudness of a sound in decibels (0 is the maximum, a quiet room is about -60)."""
    return 20 * np.log10(max(float(np.sqrt(np.mean(np.square(x)))), 1e-6))


detector = WakeDetector(oww)
print("Silence gives these scores:", detector.feed(np.zeros(RATE, np.float32)))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How long is one piece of sound (FRAME) in milliseconds? Calculate it from the numbers above:
- What score does silence give:

---
# Part B: See the Wake Word (40 min)

### Step 1: A Test Sound Without a Microphone
The Piper voice of Lab 18 can say the wake word for you, so you can test the detector without speaking. The cell makes the sound, runs the detector over it and prints the score of every 80 ms piece that is above 0.1. If Lab 18 has not made the voice yet, the cell tells you.

In [ ]:
VOICE_FILE = next(iter(glob.glob("../lab18/voices/en_US-lessac-medium.onnx") + glob.glob("voices/en_US-lessac-medium.onnx")), None)
print("Voice file:", VOICE_FILE)


def say(text, path):
    """Make speech with Piper and save it as a WAV file."""
    result = subprocess.run([sys.executable, "-m", "piper", "-m", VOICE_FILE, "-f", path], input=text.encode(), capture_output=True)
    if result.returncode != 0:
        raise SystemExit("Piper failed: " + result.stderr.decode(errors="replace")[-300:])
    return path


def resample(sound, rate):
    if rate == RATE:
        return sound
    positions = np.arange(0, len(sound), rate / RATE)
    return np.interp(positions, np.arange(len(sound)), sound).astype(np.float32)


def timeline(sound, min_score=0.1):
    """Run the detector over a sound, piece by piece. Prints the pieces with a score and returns the best score of each word."""
    detector.reset()
    best = {}
    for start in range(0, len(sound) - FRAME + 1, FRAME):
        scores = detector.feed(sound[start:start + FRAME])
        for name, score in scores.items():
            best[name] = max(best.get(name, 0.0), score)
            if score > min_score:
                print(f"  at {start / RATE:4.2f} s   {name:<12} {score:.2f}  " + "#" * int(score * 30))
    return best


if VOICE_FILE:
    wav, rate = labview.load_wav(say("Hey Jarvis.", "wake_test.wav"))
    test = np.concatenate([np.zeros(RATE, np.float32), resample(wav, rate), np.zeros(RATE, np.float32)])
    best = timeline(test)
    print("Best scores:", {k: round(v, 2) for k, v in best.items()})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which 80 ms pieces had a high score (time in seconds):
- What was the best score:
- What score do the pieces before and after the word have:

### Step 2: Other Words and Other Sentences
The detector must say *yes* to the wake word and *no* to everything else. The cell tests several sentences, including words that sound a little like the wake word. Look at the best score of each.

In [ ]:
sentences = ["Hey Jarvis.", "Hello Jarvis.", "Hey there.", "Good morning everyone.", "Turn on the light please.", "Jarvis."]
if VOICE_FILE:
    for sentence in sentences:
        wav, rate = labview.load_wav(say(sentence, "wake_test.wav"))
        sound = np.concatenate([np.zeros(RATE, np.float32), resample(wav, rate), np.zeros(RATE, np.float32)])
        detector.reset()
        top = max(max(detector.feed(sound[s:s + FRAME]).values()) for s in range(0, len(sound) - FRAME + 1, FRAME))
        print(f"{sentence:<28} best score {top:.2f}   {'WAKE' if top >= THRESHOLD else '-'}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which sentences woke the detector:
- Which sentence that is not the wake word came closest, and what score:
- Would a different THRESHOLD change this?:

### Step 3: Your Voice
Now use the microphone. The cell records 6 seconds. After one second of quiet say **the wake word** once, in your normal voice. Then it prints the scores.

In [ ]:
print("Stay quiet for a moment ...")
time.sleep(1)
print(">>> SAY THE WAKE WORD ONCE, AFTER ONE SECOND <<<")
mine = labview.record(6)
print("Level of the recording:", round(level_db(mine), 1), "dB")
best = timeline(mine)
print("Best scores:", {k: round(v, 2) for k, v in best.items()})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Best score of the wake word with your voice:
- Did it pass THRESHOLD? What would you change if not (speak louder, closer, or lower THRESHOLD):

---
# Part C: Listen Live and Tune the Threshold (35 min)

### Step 1: The Live Listener
The class `WakeListener` reads the microphone all the time in a background thread. It counts how much new sound arrived since the last look, gives it to the detector, and shows the **score of each wake word** as a bar. When a score is above `THRESHOLD` (and the cooldown has passed) it counts one *wake* and writes it in `wake_log.csv`. Run the cell, say the wake word several times, and talk normally. Press **Stop** when you are done.

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def log_wake(word, score):
    first = not os.path.exists("wake_log.csv")
    with open("wake_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "word", "score"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), word, round(score, 2)])


class WakeListener:
    def __init__(self):
        self.stream = labview.MicStream(RATE)
        self.detector = WakeDetector(Model(wakeword_models=WAKE_WORDS, inference_framework="onnx"))
        self.last_time = time.time()
        self.last_wake = 0.0
        self.wakes = 0
        self.stop_flag = threading.Event()
        self.screen = widgets.HTML("Listening ...")
        self.button = widgets.Button(description="Stop", button_style="danger")
        self.button.on_click(lambda _: self.stop())
        display(widgets.VBox([self.button, self.screen]))
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while not self.stop_flag.is_set():
            time.sleep(0.08)
            if not self.stream.alive():
                self.screen.value = "The microphone stopped: " + self.stream.error()
                break
            now = time.time()
            seconds = min(now - self.last_time, 2.0)
            self.last_time = now
            scores = self.detector.feed(self.stream.latest(seconds))
            for word, score in scores.items():
                if score >= THRESHOLD and now - self.last_wake >= COOLDOWN:
                    self.last_wake = now
                    self.wakes += 1
                    log_wake(word, score)
            bars = "".join(f"{w:<12} {s:4.2f} " + "#" * int(s * 30) + "<br>" for w, s in scores.items())
            self.screen.value = f"<pre>{bars}\nthreshold {THRESHOLD}   wake-ups: {self.wakes}   loudness {level_db(self.stream.latest(0.2)):.0f} dB</pre>"

    def stop(self):
        self.stop_flag.set()
        self.stream.stop()
        self.screen.value += "   [stopped]"


listener = WakeListener()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many times did you say the wake word, and how many wake-ups were counted:
- Did it wake when you said other words? Which:

### Step 2: Tune the Threshold
Change `THRESHOLD` in the settings cell of Part A (try 0.3, 0.5 and 0.8), run that cell, and start the listener again (press Stop first). At each value say the wake word 10 times, and then talk normally (read a text aloud) for one minute. Fill in the table (double-click to edit).

| THRESHOLD | Wake word said 10 times: heard | Talking for 1 minute: false wake-ups |
|---|---|---|
| 0.3 | | |
| 0.5 | | |
| 0.8 | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which THRESHOLD is best for a voice assistant at home and why:
- Which mistake is worse: a missed wake word or a false wake-up? Explain for a hospital bed call button and for a toy:

After you pressed Stop, see the log:

In [ ]:
if os.path.exists("wake_log.csv"):
    with open("wake_log.csv") as f:
        print(f.read())
else:
    print("No wake-ups yet.")

---
# Part D: The Voice-Controlled Assistant (60 min)

### Step 1: The Ears and the Mouth
The assistant needs two more tools that you know. **Whisper** (Lab 17) turns the command into text. **Piper** (Lab 18) says the answer. Run the cell to load both. The first call of Piper makes the voice file and keeps every sentence in a cache, so repeated sentences are instant.

In [ ]:
from faster_whisper import WhisperModel

stt = WhisperModel(STT_SIZE, device="cpu", compute_type="int8", cpu_threads=4)
os.makedirs("speech_cache", exist_ok=True)


def transcribe(sound):
    segments, info = stt.transcribe(sound.astype(np.float32), language="en", beam_size=1)
    return " ".join(s.text.strip() for s in segments).strip()


def speak(text, browser=True):
    """Text to speech with Piper. Returns the time until the sound was ready."""
    started = time.time()
    path = f"speech_cache/{hashlib.md5(text.encode()).hexdigest()[:12]}.wav"
    if not os.path.exists(path):
        say(text, path)
    made = time.time() - started
    labview.play(path, browser=browser)
    return made


print("Whisper", STT_SIZE, "loaded. Voice:", VOICE_FILE)

### Step 2: The Commands
The assistant knows a few **fixed commands**. The function `reply_to` looks for words in the text, does the action and returns the answer. The light is a *pretend* light (a variable). In Lab 27 the same idea switches a real LED on the ESP32.

| You say | Action | Answer |
|---|---|---|
| "turn on the light" | `light = True` | "The light is on." |
| "turn off the light" | `light = False` | "The light is off." |
| "what time is it" | reads the clock | "It is 10 45." |
| "hello" | nothing | "Hello. How can I help?" |
| "goodbye" or "stop listening" | stops the assistant | "Goodbye." |
| anything else | nothing | "Sorry, I do not know that command." |

In [ ]:
import re

home = {"light": False, "stop": False}


def reply_to(text):
    """Find the command in the text, do the action and return the answer (a sentence)."""
    t = re.sub(r"[^a-z ]", " ", text.lower())
    words = set(t.split())
    if "light" in words and "on" in words:
        home["light"] = True
        return "The light is on."
    if "light" in words and "off" in words:
        home["light"] = False
        return "The light is off."
    if "time" in words:
        return datetime.now().strftime("It is %H %M.")
    if words & {"goodbye", "bye"} or "stop listening" in t:
        home["stop"] = True
        return "Goodbye."
    if words & {"hello", "hi", "hey"}:
        return "Hello. How can I help?"
    return "Sorry, I do not know that command."


for sentence in ["Turn on the light.", "Please switch the light off.", "What time is it?", "Hello there", "Sing me a song", "Goodbye"]:
    print(f"{sentence:<30} -> {reply_to(sentence)}")
home.update(light=False, stop=False)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which sentence was understood by words that are not exactly in the table:
- Which sentence would a person understand but the program does not (write one):
- How would you add the command: what is the temperature:

### Step 3: Listen Until You Are Quiet
After the wake word the assistant records **until you stop speaking**. `listen_until_silence` reads the microphone stream every 0.1 s, waits for speech to start, and stops when you have been quiet for `SILENCE` seconds (or after `MAX_COMMAND` seconds).

In [ ]:
SILENCE = 1.0          # stop after this many seconds of quiet
MAX_COMMAND = 6.0      # never record longer than this


def listen_until_silence(stream, wait_for_speech=5.0, now=time.time, sleep=time.sleep):
    """Record from the microphone stream until you were quiet for SILENCE seconds after speaking. Returns the sound (empty if nobody spoke)."""
    chunks, spoke, last_loud, begin, started = [], False, 0.0, 0.0, now()
    while now() - started < MAX_COMMAND + wait_for_speech:
        sleep(0.1)
        piece = stream.latest(0.1)
        chunks.append(piece)
        if level_db(piece) > LIMIT_DB:
            if not spoke:
                spoke, begin = True, now()
            last_loud = now()
        if spoke and now() - last_loud >= SILENCE:
            break
        if spoke and now() - begin >= MAX_COMMAND:
            break
        if not spoke and now() - started >= wait_for_speech:
            return np.zeros(0, np.float32)
    return np.concatenate(chunks) if spoke else np.zeros(0, np.float32)


def handle_command(sound):
    """Understand a command sound and answer it. Returns (heard text, answer, seconds for each step)."""
    t0 = time.time()
    heard = transcribe(sound) if len(sound) else ""
    t1 = time.time()
    answer = reply_to(heard) if heard else "I did not hear a command."
    made = speak(answer)
    return heard, answer, {"understand": t1 - t0, "voice": made}


def log_turn(wake, heard, answer, times):
    first = not os.path.exists("assistant_log.csv")
    with open("assistant_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "wake", "heard", "answer", "understand_s", "voice_s"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), wake, heard, answer, round(times["understand"], 2), round(times["voice"], 2)])

### Step 4: Test the Command Step with a Voice Made by Piper
Before the live test, check the second half of the assistant (understand, act, speak) with a command that Piper says. This tests Whisper, `reply_to` and the voice without using the microphone.

In [ ]:
if VOICE_FILE:
    wav, rate = labview.load_wav(say("Turn on the light.", "command_test.wav"))
    command = resample(wav, rate)
    heard, answer, times = handle_command(command)
    print("Heard:  ", repr(heard))
    print("Answer: ", repr(answer))
    print("Light is on:", home["light"], "| understand", round(times["understand"], 1), "s | voice", round(times["voice"], 1), "s")
    home["light"] = False

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did Whisper hear:
- How long did understanding and the voice take:

### Step 5: Run the Assistant
Stop the listener of Part C first. Then run the cell. Say the **wake word**, wait for the beep, and say a command, for example *turn on the light*. The assistant answers with the voice and shows the light state. The button **Talk** is the **push-to-talk** fallback: press it instead of saying the wake word. Say *goodbye* or press **Stop** to end.

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def beep():
    t = np.arange(int(0.15 * RATE)) / RATE
    labview.play((0.3 * np.sin(2 * np.pi * 880 * t) * np.hanning(len(t))).astype(np.float32), browser=False)


class Assistant:
    def __init__(self):
        self.stream = labview.MicStream(RATE)
        self.detector = WakeDetector(Model(wakeword_models=WAKE_WORDS, inference_framework="onnx"))
        self.last_time = time.time()
        self.last_wake = 0.0
        self.talk = threading.Event()
        self.stop_flag = threading.Event()
        self.lines = ["Say the wake word ..."]
        self.screen = widgets.HTML("Say the wake word ...")
        self.talk_button = widgets.Button(description="Talk", button_style="info")
        self.talk_button.on_click(lambda _: self.talk.set())
        self.stop_button = widgets.Button(description="Stop", button_style="danger")
        self.stop_button.on_click(lambda _: self.stop())
        display(widgets.VBox([widgets.HBox([self.talk_button, self.stop_button]), self.screen]))
        home.update(stop=False)
        threading.Thread(target=self._run, daemon=True).start()

    def show(self, line):
        self.lines = (self.lines + [line])[-8:]
        self.screen.value = "<br>".join(self.lines) + f"<br><b>LIGHT: {'ON' if home['light'] else 'off'}</b>"

    def _wake_word(self):
        """Returns the wake word when it was heard (or "button" for push-to-talk), otherwise None."""
        now = time.time()
        seconds = min(now - self.last_time, 2.0)
        self.last_time = now
        scores = self.detector.feed(self.stream.latest(seconds))
        for word, score in scores.items():
            if score >= THRESHOLD and now - self.last_wake >= COOLDOWN:
                self.last_wake = now
                return word
        if self.talk.is_set():
            self.talk.clear()
            return "button"
        return None

    def _run(self):
        while not self.stop_flag.is_set() and not home["stop"]:
            time.sleep(0.08)
            if not self.stream.alive():
                self.show("The microphone stopped: " + self.stream.error())
                break
            wake = self._wake_word()
            if wake is None:
                continue
            self.show(f"Wake: {wake}. Speak now ...")
            beep()
            sound = listen_until_silence(self.stream)
            heard, answer, times = handle_command(sound)
            log_turn(wake, heard, answer, times)
            self.show(f"You: {heard or '(nothing)'}")
            self.show(f"Assistant: {answer}")
            self.detector.reset()
            self.last_time = time.time()
        self.stop()

    def stop(self):
        self.stop_flag.set()
        self.stream.stop()
        self.show("[stopped]")


assistant = Assistant()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which commands worked at the first try:
- Which command failed? What did Whisper hear instead:
- How many seconds passed from the end of your command to the start of the answer:

After you pressed Stop, see the log:

In [ ]:
if os.path.exists("assistant_log.csv"):
    with open("assistant_log.csv") as f:
        print(f.read())
else:
    print("No turns yet.")

---
# Part E: Make It Your Own, Test and Present (35 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the assistant again.

- Add three new commands (for example *what day is it*, *tell me a joke*, *turn on the fan*), each with its own answer.
- Use a second wake word and say a different answer for each one.
- Make `reply_to` more flexible: accept *switch*, *turn* and *put* in front of *on* and *off*.
- Keep a count of the commands in a dictionary and add the command *how many commands*.
- Repeat the heard text before the answer ("I heard turn on the light. The light is on.").

> **Tip.** Change one thing at a time and run it after each change.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- Did it work the first time? What went wrong:

### Step 2: Test
Stop the assistant and delete the old logs. Then test every situation with the assistant running and write what happened.

In [ ]:
for name in ("wake_log.csv", "assistant_log.csv"):
    if os.path.exists(name):
        os.remove(name)
print("Clean.")

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | Wake word, then *turn on the light*, 30 cm | Light ON, spoken answer | |
| 2 | Wake word, then *turn off the light* | Light off | |
| 3 | Wake word said softly | Heard or not? | |
| 4 | Wake word from 2 metres | Heard or not? Write the distance at which it fails | |
| 5 | A different person says the wake word | Heard or not? | |
| 6 | Music or a video plays in the room | False wake-ups? | |
| 7 | A command that is not in the list | "Sorry, I do not know that command." | |
| 8 | Wake word, then nothing | "I did not hear a command." | |
| 9 | Push-to-talk button | Works without the wake word | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 3: what happened:
- Test 4 to 6: what happened:
- Test 7 to 9: what happened:
- The best THRESHOLD for your room:

### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration** of the **Voice-Controlled Assistant**: wake it with the wake word, switch the light on and off, ask the time, show a command it does not know, use the Talk button, and show the log.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Wake word, THRESHOLD and COOLDOWN used:
- Commands that the assistant knows (including your new ones):
- Time from the end of the command to the answer:
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future (hint: Lab 20):

> **Where next.** The Jetson can now hear a wake word, understand a command, act and speak. The commands are fixed: the assistant only knows what you wrote. In Lab 20 you give the Jetson a **language model**, so it can understand any sentence, and in Lab 22 you join vision, speech and language. If you want a wake word of your own, Lab 33 teaches you to train one.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The detector reads the sound in pieces of 80 ms and shows the score | |
| 2 | The wake word is detected from the Piper test sound, and other sentences do not wake it | |
| 3 | The live listener shows the score bar and counts the wake-ups with your voice | |
| 4 | The threshold table is complete and you explain the best value | |
| 5 | The commands are tested in Step 2 and the Piper test of the command step works | |
| 6 | The assistant wakes up, listens, answers with the voice and switches the pretend light | |
| 7 | The Talk button (push-to-talk) works | |
| 8 | At least one improvement works and you explain the code you changed | |
| 9 | The test table is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Why does a voice assistant use a small wake-word model and not Whisper all the time?:
- 2. What is a score? What does 0.9 mean and what does 0.2 mean?:
- 3. What does `THRESHOLD` do? What happens if you set it to 0.99, and to 0.05?:
- 4. Why does the detector read the sound in pieces of exactly 80 ms (1280 numbers)?:
- 5. Name the five steps of the voice assistant and the tool for each.:
- 6. Why does the assistant record until you are quiet and not for a fixed time?:
- 7. What is the weakness of fixed commands? How does a language model help?:
- 8. Why is it important that the assistant has a push-to-talk button?:
- 9. A wake-word model was trained on English. What happens with a word in another language or a strong accent?:
- 10. Where does the sound go when the listener runs? Is this safe for privacy? Explain.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Cannot record ...` | Wrong microphone | Set it again as in Lab 17 (`labview.audio_devices()`, then `labview.mic_set(...)`) |
| `No module named 'sklearn'` or `'openwakeword'` | Step 2 of Part A was not run, or there is no internet | Run both cells of Step 2 again. Check the internet with your teacher. |
| `download_models` fails | The network dropped | Run the cell again. It tries three times. |
| `NO_SUCHFILE` or the model cannot be loaded | The model files of your wake word were not downloaded | Run the cell of Step 2 again, with the name in `WAKE_WORDS` spelled exactly |
| `Voice file: None` | The Piper voice of Lab 18 is not here | Run Lab 18 first (it downloads the voice), or ask your teacher |
| The wake word is never heard | Too quiet, too far, or the wrong wake word | Speak louder and closer; lower `THRESHOLD`; check the loudness shown in the listener |
| The wake word is heard all the time | `THRESHOLD` is too low or there is loud music or talking | Raise `THRESHOLD` (0.6 or 0.7) |
| Whisper hears the wrong words | Noise, or you spoke too fast or too far | Speak clearly, closer to the microphone. Try the model small.en (set STT_SIZE) if there is time. |
| The microphone is used by another program | Another listener or notebook still runs | Press Stop in the other cells and shut down the other kernels |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(19)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- openWakeWord: https://github.com/dscripka/openWakeWord
- Whisper (faster-whisper): https://github.com/SYSTRAN/faster-whisper
- Piper voices: https://github.com/rhasspy/piper